# jlens-audit — persistent kernel

**Rule**: the SETUP cell runs ONCE. Never restart the kernel without human validation. Every experiment writes to `results/` and `figs/`.

In [9]:
import sys, os, torch
sys.path.insert(0, os.path.abspath('..'))
from transformers import AutoTokenizer
from src.config import MODEL_DIR, LENS_DIR, D_MODEL, TARGET_LAYER, SKIP_FIRST

tok = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

def _ids(text):
    s = tok.apply_chat_template([{"role": "user", "content": text}], tokenize=False,
                                add_generation_prompt=True, enable_thinking=False)
    return tok(s, add_special_tokens=False)["input_ids"]

a, b = _ids("alpha"), _ids("beta")
n = 0
while n < min(len(a), len(b)) and a[n] == b[n]:
    n += 1
print("prefix_len =", n, repr(tok.decode(a[:n])))

for kind in ("j-lens", "r-lens"):
    d = torch.load(LENS_DIR / kind / "lens.pt", map_location="cpu", weights_only=False)
    src = sorted(d["source_layers"])
    print(f"\n{kind}: keys={list(d)}")
    print("  d_model", d["d_model"], "| n_prompts", d["n_prompts"],
          "| J", tuple(d["J"].shape), d["J"].dtype)
    print("  source_layers", src[:4], "...", src[-3:], "| n =", len(src))
    print("  provenance", d["provenance"])
    del d

prefix_len = 3 '<|im_start|>user\n'

j-lens: keys=['J', 'n_prompts', 'source_layers', 'd_model', 'provenance']


AttributeError: 'dict' object has no attribute 'shape'

In [8]:
d = torch.load(LENS_DIR / "j-lens" / "lens.pt", map_location="cpu", weights_only=False)
J = d["J"]
ks = list(J)
print("type", type(J), "| n =", len(ks))
print("clés", ks[:5], "...", ks[-3:], "| type de clé", type(ks[0]))

v = J[ks[0]]
print("valeur:", type(v), getattr(v, "shape", None), getattr(v, "dtype", None))
if isinstance(v, dict):
    print("  sous-clés", list(v))

src = sorted(d["source_layers"])
print("\nsource_layers", src[:4], "...", src[-3:], "| n =", len(src))
print("d_model", d["d_model"], "| n_prompts", d["n_prompts"])
print("provenance", d["provenance"])
print("\nclés J == source_layers ?", sorted(ks) == src)

type <class 'dict'> | n = 63
clés [0, 1, 2, 3, 4] ... [60, 61, 62] | type de clé <class 'int'>
valeur: <class 'torch.Tensor'> torch.Size([5120, 5120]) torch.float16

source_layers [0, 1, 2, 3] ... [60, 61, 62] | n = 63
d_model 5120 | n_prompts 25
provenance {'model_id': 'Qwen/Qwen3.6-27B', 'dataset_id': 'NeelNanda/pile-10k', 'target_layer': 62, 't_max': 128, 'n_prompts': 25, 'docs_consumed': 25, 'n_positions': 0.0, 'git_commit': 'modal', 'skip_first': 4, 'config_json': '{"estimator": "standard"}', 'weighting': 'uniform', 'corpus_mode': 'pretrain'}

clés J == source_layers ? True


In [3]:
d = torch.load(LENS_DIR/"j-lens"/"lens.pt", map_location="cpu", weights_only=False)
print(d["d_model"], d["n_prompts"], sorted(d["source_layers"])[:6], d["provenance"])

5120 25 [0, 1, 2, 3, 4, 5] {'model_id': 'Qwen/Qwen3.6-27B', 'dataset_id': 'NeelNanda/pile-10k', 'target_layer': 62, 't_max': 128, 'n_prompts': 25, 'docs_consumed': 25, 'n_positions': 0.0, 'git_commit': 'modal', 'skip_first': 4, 'config_json': '{"estimator": "standard"}', 'weighting': 'uniform', 'corpus_mode': 'pretrain'}


In [10]:
d = torch.load(LENS_DIR / "r-lens" / "lens.pt", map_location="cpu", weights_only=False)
print(len(d["J"]), sorted(d["J"])[:3], sorted(d["J"])[-3:], d["J"][62].dtype)
print(d["provenance"])
del d

63 [0, 1, 2] [60, 61, 62] torch.float16
{'model_id': 'Qwen/Qwen3.6-27B', 'dataset_id': 'NeelNanda/pile-10k', 'target_layer': 62, 't_max': 128, 'n_prompts': 25, 'docs_consumed': 25, 'n_positions': 0.0, 'git_commit': 'modal', 'skip_first': 4, 'config_json': '{"estimator": "relp", "rules": {"ln_rule": true, "identity_rule": true, "half_rule": true, "include_qk_norms": false}}', 'weighting': 'uniform', 'corpus_mode': 'pretrain'}


In [11]:
for kind in ("j-lens", "r-lens"):
    d = torch.load(LENS_DIR / kind / "lens.pt", map_location="cpu", weights_only=False)
    J, src = d["J"], sorted(d["source_layers"])
    ks = sorted(J)
    print(f"\n{kind}")
    print("  d_model", d["d_model"], "| n_prompts", d["n_prompts"])
    print("  J: dict de", len(ks), "couches", ks[:3], "...", ks[-3:],
          "| tenseur", tuple(J[ks[0]].shape), J[ks[0]].dtype)
    print("  clés J == source_layers ?", ks == src)
    print("  provenance", d["provenance"])
    del d


j-lens
  d_model 5120 | n_prompts 25
  J: dict de 63 couches [0, 1, 2] ... [60, 61, 62] | tenseur (5120, 5120) torch.float16
  clés J == source_layers ? True
  provenance {'model_id': 'Qwen/Qwen3.6-27B', 'dataset_id': 'NeelNanda/pile-10k', 'target_layer': 62, 't_max': 128, 'n_prompts': 25, 'docs_consumed': 25, 'n_positions': 0.0, 'git_commit': 'modal', 'skip_first': 4, 'config_json': '{"estimator": "standard"}', 'weighting': 'uniform', 'corpus_mode': 'pretrain'}

r-lens
  d_model 5120 | n_prompts 25
  J: dict de 63 couches [0, 1, 2] ... [60, 61, 62] | tenseur (5120, 5120) torch.float16
  clés J == source_layers ? True
  provenance {'model_id': 'Qwen/Qwen3.6-27B', 'dataset_id': 'NeelNanda/pile-10k', 'target_layer': 62, 't_max': 128, 'n_prompts': 25, 'docs_consumed': 25, 'n_positions': 0.0, 'git_commit': 'modal', 'skip_first': 4, 'config_json': '{"estimator": "relp", "rules": {"ln_rule": true, "identity_rule": true, "half_rule": true, "include_qk_norms": false}}', 'weighting': 'uniform'

In [12]:
d = torch.load(LENS_DIR / "j-lens" / "lens.pt", map_location="cpu", weights_only=False)
J62 = d["J"][62].float()
I = torch.eye(J62.shape[0])
print("ancre = I ?", torch.allclose(J62, I, atol=1e-3))
print("diagonale moyenne", J62.diagonal().mean().item(),
      "| hors-diagonale max", (J62 - I).abs().max().item())
del d, J62, I

ancre = I ? True
diagonale moyenne 1.0 | hors-diagonale max 0.0


In [ ]:
# SETUP — once only
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.load_model import load, layers, get_resid
tok, model = load()
print('model loaded —', model.config.num_hidden_layers, 'layers ; scanned:', layers())

In [ ]:
from src.load_model import load, layers, get_resid
tok, model = load()
print(model.config.num_hidden_layers, "couches | grille :", layers())
print(f"VRAM {torch.cuda.memory_allocated()/1e9:.1f} Go")

In [ ]:
# LENSES — after resolving the # ADAPTER points in src/lens.py by reading lenses/README.md
from src.lens import load_all
lenses = load_all()
print({k: len(v.maps) for k, v in lenses.items()})

## Step 2 — lens go/no-go, then soft conformity
**Go/no-go (binary, decisive)**: `identity_check` (the anchor `J_62 = I` must reproduce the logit lens) + `orientation_check` (overlap at the sub-anchor: catches a transpose, which the anchor cannot see). If either breaks → STOP, the lens setup is wrong.

**Soft conformity** ("sushi → Japan"), order of magnitude: R-lens from the early layers, J-lens later, logit lens late or never.

In [ ]:
from src.validate import identity_check, orientation_check, smoke
identity_check(lenses)      # STOP on failure
orientation_check(lenses)   # STOP on failure
smoke(lenses)

## Step 3 — quantitative validation (multihop pass@k)

In [ ]:
from src.validate import pass_at_k
pass_at_k(lenses)   # -> results/validation_multihop.json, figs/validation_multihop.png

## Step 4 — model capability on the pilot pairs (test 2)

In [ ]:
from src import capability
capability.main('pairs_pilot.jsonl')   # target >= 80% in-the-clear detection per family

## Step 5 — timing test on one pair (test 3)

In [ ]:
import json, time
from src.scan import scan_text
p = json.loads(open('../data/pairs_pilot.jsonl').readline())
t0 = time.time(); out, toks = scan_text(p['anomalous'], lenses); print(f'{time.time()-t0:.1f}s for {len(toks)} positions')
# inspect a few positions around the anomaly by hand:
from src.serialize import serialize
print('\n'.join(serialize(out['jlens']).split('\n')[:15]))

## Log
Before closing: an entry in `experiments.md` (done / verified / doubt / next step).